# Fine-tuning Cosmos-Embed1 with MSR-VTT

[Cosmos-Embed1](https://huggingface.co/nvidia/Cosmos-Embed1-448p) is a joint video-text embedder tailored for physical AI. It can be used for:

- **Text-to-video retrieval** — find videos matching a text query
- **Video-to-video search** — find videos similar to a query video
- **Zero-shot and kNN classification** — classify videos without task-specific training
- **Semantic deduplication** — detect near-duplicate videos in a dataset

The model architecture is based on QFormer, with modifications for processing video inputs. A ViT backbone processes individual frames, temporal embeddings are added, and a QFormer cross-attention module compresses the frame features into a compact embedding. The text branch uses a BERT-based encoder. Both branches are aligned via contrastive learning.

Cosmos-Embed1 has **state-of-the-art performance** on autonomous vehicle (AV) and robotics benchmarks while maintaining competitive performance in general domains.

| Variant | Resolution | Frames | Embedding Dim |
| --- | --- | --- | --- |
| Cosmos-Embed1-224p | 224×224 | 8 | 256 |
| Cosmos-Embed1-336p | 336×336 | 8 | 768 |
| Cosmos-Embed1-448p | 448×448 | 8 | 768 |

In this notebook, we use the **224p** variant and demonstrate the full fine-tuning workflow on the [MSR-VTT](https://huggingface.co/datasets/friedrichor/MSR-VTT) test_1k split (1,000 videos, 1,000 captions). To keep the notebook lightweight, we use the same test split for training, evaluation, and inference.

![Cosmos-Embed1 Teaser](https://research.nvidia.com/labs/dir/cosmos-embed1/images/teaser_embed1.gif)

![Cosmos-Embed1 Architecture](https://research.nvidia.com/labs/dir/cosmos-embed1/images/cosmos_embed1_architecture.png)

## Learning Objectives

In this notebook, you will learn how to:

* Download and prepare the MSR-VTT test_1k split (1,000 videos) for Cosmos-Embed1
* Fine-tune a pretrained Cosmos-Embed1 model on the test split
* Evaluate the model with top-K classification metrics
* Run text-to-video similarity search (inference)
* Export the model to ONNX for deployment
* Export the model to HuggingFace format for sharing and reuse

## Table of Contents

0. [Set up env variables](#head-0)
1. [Install dependencies](#head-1)
2. [Prepare dataset and pre-trained model](#head-2)
3. [Provide training specification](#head-3)
4. [Run training](#head-4)
5. [Evaluate the trained model](#head-5)
6. [Run inference (similarity search)](#head-6)
7. [Export to ONNX](#head-7)
8. [Export to HuggingFace format](#head-8)

## 0. Set up env variables <a class="anchor" id="head-0"></a>

Set `$LOCAL_PROJECT_DIR` to your workspace. The MSR-VTT dataset will be downloaded to `$LOCAL_PROJECT_DIR/data/msrvtt`, and experiment outputs will go to `$LOCAL_PROJECT_DIR/cosmos_embed1/results`.

In [ ]:
import os

%env LOCAL_PROJECT_DIR=/path/to/your/workspace

os.environ["HOST_DATA_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "data", "msrvtt")
os.environ["HOST_MODEL_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "cosmos_embed1")
os.environ["HOST_RESULTS_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "cosmos_embed1", "results")
os.environ["HOST_SPECS_DIR"] = os.path.join(
    os.getenv("NOTEBOOK_ROOT", os.getcwd()),
    "specs"
)

In [ ]:
!mkdir -p $HOST_DATA_DIR
!mkdir -p $HOST_MODEL_DIR
!mkdir -p $HOST_SPECS_DIR
!mkdir -p $HOST_RESULTS_DIR

In [ ]:
import json
import os
mounts_file = os.path.expanduser("~/.tao_mounts.json")
tao_configs = {
   "Mounts":[
       {
           "source": os.environ["LOCAL_PROJECT_DIR"],
           "destination": "/workspace/tao-experiments"
       },
       {
           "source": os.environ["HOST_DATA_DIR"],
           "destination": "/data"
       },
       {
           "source": os.environ["HOST_MODEL_DIR"],
           "destination": "/model"
       },
       {
           "source": os.environ["HOST_SPECS_DIR"],
           "destination": "/specs"
       },
       {
           "source": os.environ["HOST_RESULTS_DIR"],
           "destination": "/results"
       }
   ],
   "DockerOptions": {
        "shm_size": "64G",
        "ulimits": {
            "memlock": -1,
            "stack": 67108864
         },
        "network": "host"
   }
}
with open(mounts_file, "w") as mfile:
    json.dump(tao_configs, mfile, indent=4)

## 1. Install dependencies <a class="anchor" id="head-1"></a>

Install the `cosmos-embed1` package. This provides the `cosmos-embed1` CLI with train, evaluate, inference, and export subcommands.

**Prerequisites:**
* python >= 3.10
* NVIDIA GPU with driver > 535+
* CUDA-compatible PyTorch

In [ ]:
# Install cosmos-embed1 (editable install from repo, or pip install from wheel)
# Uncomment the appropriate line:
# !pip install -e /path/to/cosmos-embed1  # Editable install from local repo
# !pip install cosmos-embed1              # From wheel

# Verify installation
!tao model cosmos-embed1 --help

## 2. Prepare dataset and pre-trained model <a class="anchor" id="head-2"></a>

### 2.1 Download MSR-VTT dataset

[MSR-VTT](https://huggingface.co/datasets/friedrichor/MSR-VTT) is a large-scale video description dataset containing 10K video clips and 200K captions. We use the **test_1k** split (1,000 videos, 1,000 captions) for all steps in this notebook to keep things lightweight.

The MSR-VTT dataset class (`MSRVTTDataset`) expects:
- `mp4_urls`: A glob pattern to the video files (e.g., `/data/msrvtt/video/*.mp4`)
- `metadata`: A JSON file with video captions and metadata

Each test_1k entry has one caption per video:
```json
{"video_id": "video7020", "video": "video7020.mp4", "caption": "a woman creating a fondant baby and flower", ...}
```

In [ ]:
# Download MSR-VTT test_1k metadata and videos from HuggingFace
# Source: https://huggingface.co/datasets/friedrichor/MSR-VTT
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
import os

data_dir = os.environ["HOST_DATA_DIR"]
os.makedirs(data_dir, exist_ok=True)

# Download test_1k metadata
metadata_path = os.path.join(data_dir, "msrvtt_test_1k.json")
if not os.path.exists(metadata_path):
    hf_hub_download(
        repo_id="friedrichor/MSR-VTT",
        filename="msrvtt_test_1k.json",
        repo_type="dataset",
        local_dir=data_dir,
    )
    print("Downloaded msrvtt_test_1k.json")
else:
    print("Already exists: msrvtt_test_1k.json")

# Download video archive (~2.19 GB)
video_dir = os.path.join(data_dir, "video")
zip_path = os.path.join(data_dir, "MSRVTT_Videos.zip")
if not os.path.isdir(video_dir) or len(os.listdir(video_dir)) < 1000:
    if not os.path.exists(zip_path):
        print("Downloading MSRVTT_Videos.zip (~2.19 GB)...")
        hf_hub_download(
            repo_id="friedrichor/MSR-VTT",
            filename="MSRVTT_Videos.zip",
            repo_type="dataset",
            local_dir=data_dir,
        )
    print("Extracting videos...")
    os.makedirs(video_dir, exist_ok=True)
    !unzip -q -j "$HOST_DATA_DIR/MSRVTT_Videos.zip" "video/*.mp4" -d "$HOST_DATA_DIR/video/"
    print(f"Extracted {len(os.listdir(video_dir))} videos to {video_dir}")
else:
    print(f"Videos already present: {len(os.listdir(video_dir))} files")

In [ ]:
# Verify downloaded data
import json, glob, os

data_dir = os.environ["HOST_DATA_DIR"]

metadata_path = os.path.join(data_dir, "msrvtt_test_1k.json")
with open(metadata_path) as f:
    data = json.load(f)
print(f"msrvtt_test_1k.json: {len(data)} entries")
print(f"Sample entry: {json.dumps(data[0], indent=2)}")

video_dir = os.path.join(data_dir, "video")
videos = glob.glob(os.path.join(video_dir, "*.mp4"))
print(f"\nVideo files found: {len(videos)} (need 1,000 for test_1k)")

### 2.2 Download pre-trained model

Download the pretrained [Cosmos-Embed1-224p](https://huggingface.co/nvidia/Cosmos-Embed1-224p) checkpoint from HuggingFace. This serves as the starting point for fine-tuning.

In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = os.environ["HOST_MODEL_DIR"]
snapshot_download(
    repo_id="nvidia/Cosmos-Embed1-224p",
    local_dir=os.path.join(model_dir, "Cosmos-Embed1-224p"),
)
print(f"Model downloaded to {model_dir}/Cosmos-Embed1-224p")

In [ ]:
print("Check that model is downloaded into dir.")
!ls -l $HOST_MODEL_DIR/Cosmos-Embed1-224p/

## 3. Provide training specification <a class="anchor" id="head-3"></a>

Cosmos-Embed1 uses YAML experiment specs with a typed dataclass schema. The configuration is organized as:

* **model**: Model architecture
    * `pretrained_model_path`: path to the pretrained checkpoint
    * `precision`: training precision (`bf16`, `fp16`, `fp32`)
    * `network`: embed_dim, num_video_frames, spatial_resolution, contrastive_type, etc.
    * `network.visual_encoder`: ViT backbone settings
* **dataset**: Separate configs per mode — `train_dataset`, `val_dataset`, `test_dataset`, `inference_dataset`
    * `dataset_type`: `msrvtt` for MSR-VTT videos
    * `mp4_urls`: glob pattern for video files
    * `metadata`: path to JSON metadata
    * `split`: split filter (e.g., `train`, `test`)
    * `random_caption`: randomly sample from multiple captions per video
    * `batch_size`, `workers`, `num_video_frames`, `resolution`
* **train**: Training hyperparameters
    * `num_gpus`: number of GPUs (1, >1, or -1 for auto-detect)
    * `max_iter`: max training iterations
    * `optim`: optimizer, LR, scheduler
    * `freeze_visual_encoder`, `use_captioning_loss`
    * `lora`: LoRA fine-tuning settings
    * `callbacks`: training callbacks (wandb, gradient_clip, etc.)
* **evaluate**: checkpoint, callbacks (topk_classification, embedding_visualization)
* **inference**: checkpoint, query inputs (text/video), k
* **export**: ONNX export settings (mode, batch_size, simplify)

In [ ]:
!cat $HOST_SPECS_DIR/train.yaml

### LoRA Fine-tuning (Optional)

For parameter-efficient fine-tuning, Cosmos-Embed1 supports [LoRA (Low-Rank Adaptation)](https://arxiv.org/abs/2106.09685). LoRA trains low-rank adapter matrices instead of full model weights, significantly reducing memory and training time.

Key LoRA settings:
- `train.lora.enabled: true` — enable LoRA adapters
- `train.lora.lora_rank: 8` — rank of low-rank matrices
- `train.lora.lora_alpha: 16` — scaling factor (typically 2x rank)
- `model.network.visual_encoder.transformer_engine: false` — **required** (PEFT cannot inject into Transformer Engine layers)

A ready-to-use LoRA spec is provided at `specs/train_lora.yaml`.

In [ ]:
!cat $HOST_SPECS_DIR/train_lora.yaml

## 4. Run training <a class="anchor" id="head-4"></a>

* The visual encoder (EVA-ViT-G) is frozen by default; only the Q-Former and projection layers are trained.
* Training uses mixed precision (bfloat16) for faster training on NVIDIA Ampere/Hopper GPUs.
* For multi-GPU training, set `train.num_gpus` to the desired count (the entrypoint auto-spawns via torchrun).
* Training checkpoints are saved as `cosmos_embed1_model_latest.pth` with periodic snapshots.
* We use the test_1k split for both training and validation to keep this notebook self-contained.

In [ ]:
print("For multi-GPU, change train.num_gpus in train.yaml or via the command line.")

!tao model cosmos-embed1 train \
    -e /specs/train.yaml \
    results_dir=/results

In [ ]:
print("Trained checkpoints:")
print("---------------------")
!ls -ltrh $HOST_RESULTS_DIR/train

In [ ]:
# Use the latest checkpoint for subsequent steps
os.environ["CHECKPOINT"] = os.path.join(os.getenv("HOST_RESULTS_DIR"), "train/cosmos_embed1_model_latest.pth")

# Update the ownership of the results directory to the current user
# ! sudo chown -R $(id -u):$(id -g) $HOST_RESULTS_DIR
print("Latest checkpoint:")
!ls -lh $CHECKPOINT

## 5. Evaluate the trained model <a class="anchor" id="head-5"></a>

Evaluation runs top-K classification metrics and (optionally) embedding visualization on the MSR-VTT test_1k split.

Key evaluation parameters:

* `evaluate.checkpoint`: path to the trained model checkpoint
* `evaluate.callbacks.topk_classification`: enable top-K hit rate metrics (default: true)
* `evaluate.callbacks.embedding_visualization`: enable UMAP/t-SNE visualization (default: true)
* `evaluate.callbacks.top_k_values`: K values for hit rate, e.g. `[1, 3, 5, 10]`

Evaluation also supports **embedding caching** via pkl files to speed up repeated runs:
* `evaluate.save_dataset_pkl`: save computed embeddings to a pkl file
* `evaluate.load_dataset_pkl`: load pre-computed embeddings (skips model inference)

In [ ]:
!cat $HOST_SPECS_DIR/evaluate.yaml

In [ ]:
!tao model cosmos-embed1 evaluate \
    -e /specs/evaluate.yaml \
    results_dir=/results

In [ ]:
# Optional: save embeddings for faster re-evaluation
# !tao model cosmos-embed1 evaluate \
#     -e /specs/evaluate.yaml \
#     evaluate.save_dataset_pkl=/results/evaluate/embeddings.pkl \
#     results_dir=/results

# Subsequent runs can load cached embeddings (skips model inference):
# !tao model cosmos-embed1 evaluate \
#     -e /specs/evaluate.yaml \
#     evaluate.load_dataset_pkl=/results/evaluate/embeddings.pkl \
#     results_dir=/results

## 6. Run inference (similarity search) <a class="anchor" id="head-6"></a>

Inference performs top-K similarity search against the MSR-VTT test set. Given text queries, the model retrieves the K most similar videos from the search database.

Key inference parameters:

* `inference.checkpoint`: path to the trained model checkpoint
* `inference.query.input_texts`: list of text queries
* `inference.query.input_videos`: list of video paths for video-to-video search
* `inference.k`: number of nearest neighbors to retrieve (default: 5)
* `dataset.inference_dataset`: the video corpus to search against

Results are saved to `results.json` in the output directory with similarity scores.

In [ ]:
!cat $HOST_SPECS_DIR/inference.yaml

In [ ]:
# Text-to-video search against MSR-VTT test set
# Query texts are defined in /specs/inference.yaml under inference.query.input_texts
!tao model cosmos-embed1 inference \
    -e /specs/inference.yaml \
    inference.k=5 \
    results_dir=/results

In [ ]:
# View the retrieval results
import json

results_path = os.path.join(os.environ["HOST_RESULTS_DIR"], "inference", "results.json")
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print(json.dumps(results, indent=2))
else:
    print(f"Results file not found at {results_path}")

In [ ]:
# Optional: cache the search database for fast repeated searches
# !tao model cosmos-embed1 inference \
#     -e /specs/inference.yaml \
#     inference.save_dataset_pkl=/results/inference/db_embeddings.pkl \
#     'inference.query.input_texts=["a man is singing"]' \
#     results_dir=/results

# Fast search with cached database:
# !tao model cosmos-embed1 inference \
#     -e /specs/inference.yaml \
#     inference.load_dataset_pkl=/results/inference/db_embeddings.pkl \
#     'inference.query.input_texts=["a woman playing basketball", "people dancing at a party"]' \
#     results_dir=/results

## 7. Export to ONNX <a class="anchor" id="head-7"></a>

Export the trained model to ONNX format for deployment. Cosmos-Embed1 supports three export modes:

| Mode | ONNX Inputs | ONNX Outputs |
| --- | --- | --- |
| `video` | `videos (B,3,T,H,W)` | `video_embedding (B, embed_dim)` |
| `text` | `input_ids (B,seq_len)`, `attention_mask (B,seq_len)` | `text_embedding (B, embed_dim)` |
| `combined` | all of above | both of above |

Key export parameters:
* `export.checkpoint`: trained model checkpoint
* `export.mode`: `video`, `text`, or `combined`
* `export.batch_size`: batch size for export (-1 for dynamic batch)
* `export.simplify`: apply onnxsim simplification after export

In [ ]:
!cat $HOST_SPECS_DIR/export_onnx.yaml

In [ ]:
# Export video encoder to ONNX
!tao model cosmos-embed1 export \
    -e /specs/export_onnx.yaml \
    export.checkpoint=/results/train/cosmos_embed1_model_latest.pth \
    export.onnx_file=/results/export/cosmos_embed1_video.onnx \
    results_dir=/results

In [ ]:
print("Exported ONNX models:")
!ls -lh $HOST_RESULTS_DIR/export/

## 8. Export to HuggingFace format <a class="anchor" id="head-8"></a>

In addition to ONNX, Cosmos-Embed1 can export the trained model to [HuggingFace](https://huggingface.co/) format. This produces a self-contained directory with:

* **Sharded safetensors** — model weights split into ~500 MB shards
* **`config.json`** — model configuration compatible with `AutoModel.from_pretrained()`
* **Tokenizer files** — `vocab.txt`, `tokenizer_config.json`, etc. from `bert-base-uncased`

The exported model can be loaded directly with the HuggingFace `transformers` library or uploaded to the HuggingFace Hub for sharing.

Key export parameters:
* `export.checkpoint`: trained model checkpoint (`.pth` file)
* `export.mode`: set to `huggingface`
* `export.hf_output_dir`: output directory (auto-derived from checkpoint if null)
* `export.on_cpu`: run export on CPU (recommended for portability)

In [ ]:
!cat $HOST_SPECS_DIR/export_hf.yaml

In [ ]:
# Export trained model to HuggingFace format
!tao model cosmos-embed1 export \
    -e /specs/export_hf.yaml \
    export.checkpoint=/results/train/cosmos_embed1_model_latest.pth \
    export.hf_output_dir=/results/export_hf/cosmos_embed1_hf \
    results_dir=/results

In [ ]:
print("Exported HuggingFace model:")
!ls -lh $HOST_RESULTS_DIR/export_hf/cosmos_embed1_hf/

Once exported, the HuggingFace model can be loaded with the `transformers` library:

```python
from transformers import AutoModel, AutoConfig

model = AutoModel.from_pretrained(
    "/results/export_hf/cosmos_embed1_hf",
    trust_remote_code=True,
)
```

You can also upload the exported directory to the [HuggingFace Hub](https://huggingface.co/docs/hub/repositories-getting-started) to share with others:

```bash
huggingface-cli upload my-org/my-cosmos-embed1 /results/export_hf/cosmos_embed1_hf
```

This notebook has come to an end. You have successfully:

1. Prepared the MSR-VTT test_1k dataset (1,000 videos) for Cosmos-Embed1
2. Fine-tuned the model on the test split
3. Evaluated with top-K classification metrics
4. Performed text-to-video similarity search
5. Exported the model to ONNX for deployment
6. Exported the model to HuggingFace format for sharing and reuse

For more information, refer to:
- [Cosmos-Embed1 on HuggingFace](https://huggingface.co/nvidia/Cosmos-Embed1-448p)
- [Cosmos-Embed1 Research Page](https://research.nvidia.com/labs/dir/cosmos-embed1/)
- [MSR-VTT Dataset](https://huggingface.co/datasets/friedrichor/MSR-VTT)